# 06 — MEM Electricity | Silver Transformation

Two tidy tables from the MEM annex, parsed with `src/shared/mem_parsers.py`:

| Table | Grain | Content |
|---|---|---|
| `silver.mem_ede_monthly` | month × indicator × distributor (Edenorte, Edesur, Edeeste, Total EDE) | Energy purchased/billed, average purchase and selling prices, losses, collection, CRI, customers… since 2009 |
| `silver.mem_tariffs_quarterly` | tariff period × tariff (BTS1, BTS2, BTD, BTH, MTD1, MTD2, MTH) × concept × type × distributor | Reference/indexed tariff and applied tariff in RD$, since 2020 |

| Rule | Treatment |
|---|---|
| Percent indicators | Published as fractions (0.43); stored × 100 with unit `%`. |
| Applied tariff before Oct 2021 | Published once for all distributors → `distributor = 'Todas'`. |
| Overlapping tariff periods | `abr - may 26` and `abr - jun 26` both cover April–May 2026; kept as published with `period_order`. Gold uses the later column. |
| Uniqueness | Enforced on the business key of each table; violating rows go to `quality.dq_rejected_records`. |

**Status:** Development.

In [0]:
import sys, json, uuid, importlib
from datetime import datetime, timezone
from collections import Counter
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import mem_parsers as MP
importlib.reload(MP)

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.bronze.mem_electricity_raw_rows"
T_EDE = f"{CATALOG}.silver.mem_ede_monthly"
T_TAR = f"{CATALOG}.silver.mem_tariffs_quarterly"
REJECTED = f"{CATALOG}.quality.dq_rejected_records"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

b = spark.table(SOURCE)
latest = b.agg(F.max_by("_source_file", "_ingestion_ts")).first()[0]
b = b.filter(F.col("_source_file") == latest)
b = b.join(b.groupBy().agg(F.max("_ingestion_ts").alias("_ingestion_ts")), "_ingestion_ts")
def sheet_rows(name):
    rs = b.filter(F.col("_source_sheet") == name).orderBy("_source_row_number").collect()
    width = max(r["_source_row_number"] for r in rs)
    grid = [[] for _ in range(width)]
    for r in rs:
        grid[r["_source_row_number"] - 1] = json.loads(r["row_json"])
    return grid, rs[0]["_batch_id"]

ede_rows, batch = sheet_rows("EDE's")
tar_rows, _ = sheet_rows("Nuevo Regimen tarifario")
ede = MP.parse_ede_sheet(ede_rows)
tar = MP.parse_tariff_sheet(tar_rows)
print(f"File: {latest} · EDE records: {len(ede):,} · tariff records: {len(tar):,}")

In [0]:
checks, rejects = [], []
def add(table, name, rtype, tested, failed):
    checks.append((RUN_ID, table, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))

# EDE monthly
for r in ede:
    if r["unit"] and r["unit"].strip() == "%":
        r["value"] = r["value"] * 100
key = lambda r: (r["month"], r["indicator"], r["unit"], r["distributor"])
dup = Counter(key(r) for r in ede)
seen, ede_ok = set(), []
for r in ede:
    k = key(r)
    if dup[k] > 1 and k in seen:
        rejects.append((T_EDE, "duplicate_month_indicator_distributor", str(k), r)); continue
    seen.add(k); ede_ok.append(r)
add(T_EDE, "unique_month_indicator_distributor", "enforced", len(ede), len(ede) - len(ede_ok))
dists = Counter(r["distributor"] for r in ede_ok)
add(T_EDE, "four_distributor_series", "observational", len(dists), 0 if set(dists) == {"Total EDE", "Edenorte", "Edesur", "Edeeste"} else 1)
price = [r for r in ede_ok if r["indicator"] == "Precio Medio de Venta de Energía" and r["unit"] == "DOP/kWh"]
add(T_EDE, "selling_price_dop_positive", "enforced", len(price), sum(1 for r in price if r["value"] <= 0))

# Tariffs
tkey = lambda r: (r["period_order"], r["tariff"], r["concept"], r["tariff_type"], r["distributor"])
tdup = Counter(tkey(r) for r in tar)
add(T_TAR, "unique_period_tariff_concept_type_distributor", "enforced", len(tar), sum(v - 1 for v in tdup.values() if v > 1))
add(T_TAR, "period_dates_parsed", "enforced", len(tar), sum(1 for r in tar if r["period_start"] is None))
add(T_TAR, "tariff_value_positive", "enforced", len(tar), sum(1 for r in tar if r["value"] <= 0))
periods = sorted({(r["period_order"], r["period_start"], r["period_end"]) for r in tar})
overl = sum(1 for a, b2 in zip(periods, periods[1:]) if b2[1] <= a[2])
add(T_TAR, "non_overlapping_periods", "observational", len(periods), overl)
for c in checks: print(f"  {c[6]:<4} {c[1].split('.')[-1]:<24} {c[2]:<46} failed={c[5]:,}/{c[4]:,}")
assert not [c for c in checks if c[6] == "FAIL"], "Enforced rules failed; nothing written"

In [0]:
meta = lambda df: (df.withColumn("_source_file", F.lit(latest)).withColumn("_bronze_batch_id", F.lit(batch))
                   .withColumn("_silver_run_id", F.lit(RUN_ID)).withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))
e = meta(spark.createDataFrame([(r["month"], r["indicator"], r["unit"], r["distributor"], r["value"], r["source_row"]) for r in ede_ok],
        "reference_month date, indicator string, unit string, distributor string, value double, _source_row_number int"))
e.orderBy("indicator", "distributor", "reference_month").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_EDE)
spark.sql(f"COMMENT ON TABLE {T_EDE} IS 'Silver: MEM monthly indicators of the state distributors (EDE), by distributor and total, since 2009. Percentages stored as 0-100.'")
t = meta(spark.createDataFrame([(r["period_label"], r["period_start"], r["period_end"], r["period_order"], r["tariff"], r["concept"], r["tariff_type"],
                                 r["source_tariff_type"], r["distributor"], r["value"], r["source_row"]) for r in tar],
        "period_label string, period_start date, period_end date, period_order int, tariff string, concept string, tariff_type string, "
        "source_tariff_type string, distributor string, value_dop double, _source_row_number int"))
t.orderBy("tariff", "concept", "period_order", "tariff_type", "distributor").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_TAR)
spark.sql(f"COMMENT ON TABLE {T_TAR} IS 'Silver: regulated electricity tariffs (RD$) by period, tariff, concept, type (reference/applied) and distributor, from the MEM annex (SIE resolutions), since 2020.'")
if rejects:
    spark.createDataFrame([(RUN_ID, x[0], x[1], x[2], json.dumps(x[3], default=str, ensure_ascii=False), batch, RUN_TS) for x in rejects],
        "run_id string, target_table string, rule_name string, record_key string, record_json string, source_batch_id string, detected_ts timestamp").write.mode("append").saveAsTable(REJECTED)
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

s = spark.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT indicator) i, MIN(reference_month) a, MAX(reference_month) z FROM {T_EDE}").first()
s2 = spark.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT period_order) p, MIN(period_start) a, MAX(period_end) z FROM {T_TAR}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "file": latest, "ede": {"rows": s["n"], "indicators": s["i"], "months": [str(s["a"]), str(s["z"])]},
    "tariffs": {"rows": s2["n"], "periods": s2["p"], "range": [str(s2["a"]), str(s2["z"])]}, "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks}}, ensure_ascii=False))